# LeetCode 1470 — Shuffle the Array bằng Python

**Link đề:** https://leetcode.com/problems/shuffle-the-array/
**Độ khó:** Easy · **Chủ đề:** Array

```python
class Solution:
    def shuffle(self, nums: List[int], n: int) -> List[int]:
        ...
```

### Mục tiêu học
1. Đọc cấu trúc input `[x1..xn, y1..yn]` và **ánh xạ chỉ số**: `x_i` ở `nums[i]`, `y_i` ở `nums[i + n]`; trong kết quả `x_i` ở `2i`, `y_i` ở `2i + 1`.
2. Dùng thành thạo **hai con trỏ**, `zip`, slicing có bước `a[::2]`, `a[1::2]` và **gán vào slice**.
3. Đi qua 5 lời giải: hai con trỏ (hint) → công thức chỉ số → `zip` + comprehension → gán slice → **in-place `O(1)` bộ nhớ** bằng mã hóa bit.
4. Trace từng bước, chứng minh bằng ánh xạ chỉ số, phân tích Time/Space.
5. Biến thể: phép ngược (un-shuffle), trộn hai list khác độ dài (*Merge Strings Alternately*), trộn `k` phần.

**Cách học:** đọc markdown → đoán output cell code kế tiếp → chạy → so sánh. Cell TODO chỉ để luyện, không tự chạy.

## 1. Đề bài

Cho mảng `nums` gồm `2n` phần tử theo dạng `[x1, x2, ..., xn, y1, y2, ..., yn]`. Trả về mảng dạng `[x1, y1, x2, y2, ..., xn, yn]`.

| Ví dụ | `nums` | `n` | Output |
|---|---|---:|---|
| 1 | `[2,5,1,3,4,7]` | 3 | `[2,3,5,4,1,7]` |
| 2 | `[1,2,3,4,4,3,2,1]` | 4 | `[1,4,2,3,3,2,4,1]` |
| 3 | `[1,1,2,2]` | 2 | `[1,2,1,2]` |

Ví dụ 1: `x = [2,5,1]`, `y = [3,4,7]` → xen kẽ `2,3,5,4,1,7`.

**Constraints**
- `1 <= n <= 500`
- `nums.length == 2n`
- `1 <= nums[i] <= 10^3`

Câu hỏi gợi mở:
- `y1` nằm ở chỉ số nào của `nums`? `y_i` tổng quát?
- Trong kết quả, `x_i` và `y_i` nằm ở chỉ số nào?

In [1]:
nums, n = [2, 5, 1, 3, 4, 7], 3
xs, ys = nums[:n], nums[n:]
print("x =", xs, "| y =", ys)
for i in range(n):
    print(f"i={i}: x={nums[i]} (nums[{i}]), y={nums[i+n]} (nums[{i+n}])")

x = [2, 5, 1] | y = [3, 4, 7]
i=0: x=2 (nums[0]), y=3 (nums[3])
i=1: x=5 (nums[1]), y=4 (nums[4])
i=2: x=1 (nums[2]), y=7 (nums[5])


## 2. Nền tảng Python cần dùng

### 2.1. Slicing `a[start:stop]` và chia đôi mảng

`nums[:n]` lấy `n` phần tử đầu (các `x`), `nums[n:]` lấy phần còn lại (các `y`). Slicing tạo **list mới**.

In [2]:
a = [10, 20, 30, 40, 50, 60]
print(a[:3], a[3:])
print(a[1:4])

[10, 20, 30] [40, 50, 60]
[20, 30, 40]


### 2.2. Slicing có bước `a[start::step]`

- `a[::2]` lấy các vị trí chẵn `0, 2, 4, ...`.
- `a[1::2]` lấy các vị trí lẻ `1, 3, 5, ...`.

Đây đúng là "vị trí của `x`" và "vị trí của `y`" trong **kết quả**.

In [3]:
res = ["x1", "y1", "x2", "y2", "x3", "y3"]
print(res[::2])     # ['x1', 'x2', 'x3']
print(res[1::2])    # ['y1', 'y2', 'y3']

['x1', 'x2', 'x3']
['y1', 'y2', 'y3']


### 2.3. Gán vào slice có bước

`ans[::2] = list_x` ghi `list_x` vào các vị trí chẵn của `ans`. Với slice có bước, **độ dài phải khớp** chính xác, nếu không Python báo `ValueError`.

In [4]:
ans = [0] * 6
ans[::2] = [2, 5, 1]
print(ans)
ans[1::2] = [3, 4, 7]
print(ans)
try:
    ans[::2] = [9, 9]          # thiếu 1 phần tử
except ValueError as e:
    print("Lỗi:", e)

[2, 0, 5, 0, 1, 0]
[2, 3, 5, 4, 1, 7]
Lỗi: attempt to assign sequence of size 2 to extended slice of size 3


### 2.4. `zip` và comprehension hai tầng

`zip(xs, ys)` cho các cặp `(x_i, y_i)`. Comprehension hai tầng `[v for pair in pairs for v in pair]` "làm phẳng" list các cặp thành một list.

In [5]:
pairs = list(zip([2, 5, 1], [3, 4, 7]))
print(pairs)
flat = [v for pair in pairs for v in pair]
print(flat)

[(2, 3), (5, 4), (1, 7)]
[2, 3, 5, 4, 1, 7]


Đọc comprehension hai tầng **theo thứ tự viết**, giống hai vòng `for` lồng nhau:

```python
flat = []
for pair in pairs:      # tầng ngoài viết trước
    for v in pair:      # tầng trong viết sau
        flat.append(v)
```

### 2.5. Phép toán bit `<<`, `>>`, `&`, `|` (dùng cho cách in-place)

- `a << 10` = `a * 2^10 = a * 1024` (dịch trái 10 bit).
- `a >> 10` = `a // 1024`.
- `a & 1023` lấy **10 bit thấp** (= `a % 1024`).
- `a | b` "hợp" các bit (khi bit không chồng nhau thì giống `a + b`).

Vì `nums[i] <= 1000 < 1024 = 2^10`, mỗi giá trị nằm gọn trong 10 bit → có thể **nhét hai giá trị vào một số**.

In [6]:
x, y = 5, 3
packed = x | (y << 10)
print("packed =", packed, "=", x, "+", y, "* 1024")
print("lấy x:", packed & 1023, "| lấy y:", packed >> 10)
print(bin(1023), 1000 < 1024)

packed = 3077 = 5 + 3 * 1024
lấy x: 5 | lấy y: 3
0b1111111111 True


## 3. Cách 1 — Hai con trỏ (theo hint)

Hint: con trỏ `i` bắt đầu ở đầu (`x1`), con trỏ `j` bắt đầu ở vị trí `n` (`y1`). Xen kẽ lấy `nums[i]`, `nums[j]` rồi tăng cả hai.

In [7]:
def shuffle_two_pointers(nums, n):
    ans = []
    i, j = 0, n
    while i < n:
        ans.append(nums[i])
        ans.append(nums[j])
        i += 1
        j += 1
    return ans

print(shuffle_two_pointers([2, 5, 1, 3, 4, 7], 3))         # [2, 3, 5, 4, 1, 7]
print(shuffle_two_pointers([1, 2, 3, 4, 4, 3, 2, 1], 4))   # [1, 4, 2, 3, 3, 2, 4, 1]
print(shuffle_two_pointers([1, 1, 2, 2], 2))               # [1, 2, 1, 2]

[2, 3, 5, 4, 1, 7]
[1, 4, 2, 3, 3, 2, 4, 1]
[1, 2, 1, 2]


### 3.1. Nhận xét

Luôn có `j = i + n`, nên thật ra chỉ cần **một** biến `i`. Cách 1 là `O(n)` thời gian và `O(n)` bộ nhớ cho mảng kết quả — đã tối ưu về thời gian (phải ghi `2n` phần tử). Các cách sau khác nhau ở **cách viết** và **bộ nhớ phụ**.

## 4. Insight — ánh xạ chỉ số hai chiều

Nhìn bài như một **hoán vị chỉ số**:

| Phần tử | Vị trí trong `nums` | Vị trí trong `ans` |
|---|---|---|
| `x_i` (0-indexed `i`) | `i` | `2i` |
| `y_i` | `i + n` | `2i + 1` |

Ngược lại, phần tử ở vị trí `k` của `ans` đến từ: `nums[k // 2]` nếu `k` chẵn, `nums[k // 2 + n]` nếu `k` lẻ.

Kiểm tra bằng ví dụ 1:

In [8]:
nums, n = [2, 5, 1, 3, 4, 7], 3
for k in range(2 * n):
    src = k // 2 if k % 2 == 0 else k // 2 + n
    print(f"ans[{k}] lấy từ nums[{src}] = {nums[src]}")

ans[0] lấy từ nums[0] = 2
ans[1] lấy từ nums[3] = 3
ans[2] lấy từ nums[1] = 5
ans[3] lấy từ nums[4] = 4
ans[4] lấy từ nums[2] = 1
ans[5] lấy từ nums[5] = 7


## 5. Cách 2 (lời giải chính) — Công thức chỉ số, mảng cấp phát sẵn

### 5.1. Mổ xẻ

```python
ans = [0] * (2 * n)            # cấp phát sẵn 2n ô
for i in range(n):
    ans[2 * i] = nums[i]       # x_i
    ans[2 * i + 1] = nums[i + n]   # y_i
return ans
```

Mỗi vòng ghi đúng hai ô `2i`, `2i+1`; khi `i` chạy `0..n-1`, các ô `0..2n-1` được ghi **đúng một lần**.

In [9]:
def shuffle_index(nums, n):
    ans = [0] * (2 * n)
    for i in range(n):
        ans[2 * i] = nums[i]
        ans[2 * i + 1] = nums[i + n]
    return ans

print(shuffle_index([2, 5, 1, 3, 4, 7], 3))
print(shuffle_index([1, 2, 3, 4, 4, 3, 2, 1], 4))

[2, 3, 5, 4, 1, 7]
[1, 4, 2, 3, 3, 2, 4, 1]


## 6. TRACE từng bước — ví dụ 1 (`n = 3`)

| i | `nums[i]` | `nums[i+n]` | ghi vào | `ans` sau |
|---:|---:|---:|---|---|
| (đầu) | | | | `[0,0,0,0,0,0]` |
| 0 | 2 | 3 | `ans[0], ans[1]` | `[2,3,0,0,0,0]` |
| 1 | 5 | 4 | `ans[2], ans[3]` | `[2,3,5,4,0,0]` |
| 2 | 1 | 7 | `ans[4], ans[5]` | `[2,3,5,4,1,7]` |

In [10]:
def trace_shuffle(nums, n):
    ans = [0] * (2 * n)
    print("ban đầu:", ans)
    for i in range(n):
        ans[2 * i] = nums[i]
        ans[2 * i + 1] = nums[i + n]
        print(f"i={i}: x=nums[{i}]={nums[i]} → ans[{2*i}], y=nums[{i+n}]={nums[i+n]} → ans[{2*i+1}] | ans={ans}")
    return ans

trace_shuffle([2, 5, 1, 3, 4, 7], 3)

ban đầu: [0, 0, 0, 0, 0, 0]
i=0: x=nums[0]=2 → ans[0], y=nums[3]=3 → ans[1] | ans=[2, 3, 0, 0, 0, 0]
i=1: x=nums[1]=5 → ans[2], y=nums[4]=4 → ans[3] | ans=[2, 3, 5, 4, 0, 0]
i=2: x=nums[2]=1 → ans[4], y=nums[5]=7 → ans[5] | ans=[2, 3, 5, 4, 1, 7]


[2, 3, 5, 4, 1, 7]

### 6.1. Trace ví dụ 2 — dữ liệu "đối xứng" dễ làm rối mắt

In [11]:
trace_shuffle([1, 2, 3, 4, 4, 3, 2, 1], 4)

ban đầu: [0, 0, 0, 0, 0, 0, 0, 0]
i=0: x=nums[0]=1 → ans[0], y=nums[4]=4 → ans[1] | ans=[1, 4, 0, 0, 0, 0, 0, 0]
i=1: x=nums[1]=2 → ans[2], y=nums[5]=3 → ans[3] | ans=[1, 4, 2, 3, 0, 0, 0, 0]
i=2: x=nums[2]=3 → ans[4], y=nums[6]=2 → ans[5] | ans=[1, 4, 2, 3, 3, 2, 0, 0]
i=3: x=nums[3]=4 → ans[6], y=nums[7]=1 → ans[7] | ans=[1, 4, 2, 3, 3, 2, 4, 1]


[1, 4, 2, 3, 3, 2, 4, 1]

Khi giá trị lặp lại, nhìn **chỉ số** (cột `nums[...]`) thay vì nhìn giá trị để kiểm tra.

## 7. Cách 3 — `zip` + comprehension một dòng

In [12]:
def shuffle_zip(nums, n):
    return [v for pair in zip(nums[:n], nums[n:]) for v in pair]

print(shuffle_zip([2, 5, 1, 3, 4, 7], 3), shuffle_zip([1, 1, 2, 2], 2))

[2, 3, 5, 4, 1, 7] [1, 2, 1, 2]


## 8. Cách 4 — Gán vào slice có bước

In [13]:
def shuffle_slices(nums, n):
    ans = [0] * (2 * n)
    ans[::2] = nums[:n]      # vị trí chẵn ← các x
    ans[1::2] = nums[n:]     # vị trí lẻ  ← các y
    return ans

print(shuffle_slices([2, 5, 1, 3, 4, 7], 3), shuffle_slices([1, 2, 3, 4, 4, 3, 2, 1], 4))

[2, 3, 5, 4, 1, 7] [1, 4, 2, 3, 3, 2, 4, 1]


Đây là cách viết sát nhất với bảng ánh xạ ở mục 4: "chẵn nhận `x`, lẻ nhận `y`".

## 9. Cách 5 — In-place, bộ nhớ phụ `O(1)` (mã hóa bit)

Nếu **không được** tạo mảng mới (follow-up phỏng vấn thường gặp), ta tận dụng `nums[i] <= 1000 < 2^10`:

**Pha 1 — đóng gói:** với `i = 0..n-1`, nhét `y_i` vào 10 bit cao của ô `i`:
`nums[i] |= nums[i + n] << 10`. Bây giờ ô `i` chứa cả `x_i` (10 bit thấp) và `y_i` (bit cao); các ô `n..2n-1` thành "trống" (có thể ghi đè).

**Pha 2 — mở gói từ cuối về đầu:** với `i = n-1..0`:
`nums[2i + 1] = nums[i] >> 10` (`y_i`), `nums[2i] = nums[i] & 1023` (`x_i`).

Vì sao phải đi **ngược**? Ô đích `2i, 2i+1` luôn `>= i`. Đi từ `i` lớn xuống, các ô đích chỉ ghi đè những ô gói đã được mở (chỉ số `> i`) hoặc chính ô `i` — không phá ô gói `j < i` nào chưa mở.

In [14]:
def shuffle_inplace(nums, n):
    for i in range(n):
        nums[i] |= nums[i + n] << 10
    for i in range(n - 1, -1, -1):
        y = nums[i] >> 10
        x = nums[i] & 1023
        nums[2 * i + 1] = y
        nums[2 * i] = x
    return nums

data = [2, 5, 1, 3, 4, 7]
print(shuffle_inplace(data, 3), "| data cũng đã đổi:", data)

[2, 3, 5, 4, 1, 7] | data cũng đã đổi: [2, 3, 5, 4, 1, 7]


In [15]:
def trace_inplace(nums, n):
    nums = nums[:]
    for i in range(n):
        nums[i] |= nums[i + n] << 10
    print("sau pha 1:", nums, "→ (x, y) trong ô 0..n-1:", [(v & 1023, v >> 10) for v in nums[:n]])
    for i in range(n - 1, -1, -1):
        y, x = nums[i] >> 10, nums[i] & 1023
        nums[2 * i + 1] = y
        nums[2 * i] = x
        print(f"i={i}: mở ({x}, {y}) → ghi ô {2*i}, {2*i+1} | nums={nums}")
    return nums

trace_inplace([2, 5, 1, 3, 4, 7], 3)

sau pha 1: [3074, 4101, 7169, 3, 4, 7] → (x, y) trong ô 0..n-1: [(2, 3), (5, 4), (1, 7)]
i=2: mở (1, 7) → ghi ô 4, 5 | nums=[3074, 4101, 7169, 3, 1, 7]
i=1: mở (5, 4) → ghi ô 2, 3 | nums=[3074, 4101, 5, 4, 1, 7]
i=0: mở (2, 3) → ghi ô 0, 1 | nums=[2, 3, 5, 4, 1, 7]


[2, 3, 5, 4, 1, 7]

`range(n - 1, -1, -1)` đếm ngược `n-1, n-2, ..., 0` (dừng **trước** `-1`, bước `-1`).

Lưu ý: cách này dựa vào **giới hạn giá trị** của đề; nó sửa input và khó đọc hơn, nên chỉ dùng khi đề yêu cầu bộ nhớ `O(1)`.

## 10. Code nộp LeetCode (Python 3)

In [16]:
from typing import List

class Solution:
    def shuffle(self, nums: List[int], n: int) -> List[int]:
        ans = [0] * (2 * n)
        for i in range(n):
            ans[2 * i] = nums[i]
            ans[2 * i + 1] = nums[i + n]
        return ans

sol = Solution()
print(sol.shuffle([2, 5, 1, 3, 4, 7], 3))         # [2, 3, 5, 4, 1, 7]
print(sol.shuffle([1, 2, 3, 4, 4, 3, 2, 1], 4))   # [1, 4, 2, 3, 3, 2, 4, 1]
print(sol.shuffle([1, 1, 2, 2], 2))               # [1, 2, 1, 2]

[2, 3, 5, 4, 1, 7]
[1, 4, 2, 3, 3, 2, 4, 1]
[1, 2, 1, 2]


Bản ngắn (cũng Accepted):

```python
class Solution:
    def shuffle(self, nums: List[int], n: int) -> List[int]:
        ans = [0] * (2 * n)
        ans[::2], ans[1::2] = nums[:n], nums[n:]
        return ans
```

## 11. Vì sao thuật toán đúng?

Yêu cầu: `ans = [x1, y1, ..., xn, yn]`, tức (0-indexed) `ans[2i] = x_i = nums[i]` và `ans[2i+1] = y_i = nums[i+n]` với mọi `0 <= i < n`.

- Cách 2 gán **đúng** hai đẳng thức này cho mỗi `i`.
- Các chỉ số `{2i, 2i+1 : 0 <= i < n}` phủ **đúng một lần** mọi ô `0..2n-1` (mỗi `k` có duy nhất `i = k // 2` và tính chẵn/lẻ cho biết là `x` hay `y`) → không ô nào bị bỏ trống hoặc ghi hai lần.

**Invariant** (sau vòng `i`): `ans[0..2i+1]` đã đúng là `[x_0, y_0, ..., x_i, y_i]`.

Cách 5 (in-place): sau pha 1, ô `j < n` chứa `x_j + 1024·y_j`. Pha 2 ở bước `i` ghi các ô `2i, 2i+1 >= i`. Mọi ô gói `j < i` chưa mở có `j < i <= 2i` nên không bị ghi đè; ô `i` bị ghi đè **sau khi** đã đọc xong. Vậy mọi cặp được mở đúng.

## 12. Phân tích Time / Space

Gọi `n` như đề (mảng dài `2n`).

| Cách | Time | Space phụ (không tính output) | Ghi chú |
|---|---|---|---|
| 1. Hai con trỏ | `O(n)` | `O(1)` | theo hint, dùng `append` |
| 2. Công thức chỉ số | `O(n)` | `O(1)` | **khuyên dùng**, rõ ràng |
| 3. `zip` + comprehension | `O(n)` | `O(n)` (hai slice + tuple) | một dòng |
| 4. Gán slice | `O(n)` | `O(n)` (hai slice) | sát bảng ánh xạ |
| 5. In-place bit | `O(n)` | `O(1)`, **không cần mảng output** | phụ thuộc `nums[i] < 1024`, sửa input |

Với cách 1–4, mảng kết quả `O(n)` là bắt buộc vì đề yêu cầu trả mảng mới.

## 13. Bộ test — mẫu đề, biên, phản ví dụ, đối chiếu

In [17]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS  {name}: {actual}")

sol = Solution()
check("ví dụ 1", sol.shuffle([2, 5, 1, 3, 4, 7], 3), [2, 3, 5, 4, 1, 7])
check("ví dụ 2", sol.shuffle([1, 2, 3, 4, 4, 3, 2, 1], 4), [1, 4, 2, 3, 3, 2, 4, 1])
check("ví dụ 3", sol.shuffle([1, 1, 2, 2], 2), [1, 2, 1, 2])
check("n = 1", sol.shuffle([7, 9], 1), [7, 9])
check("giá trị biên 1000", sol.shuffle([1000, 1, 1, 1000], 2), [1000, 1, 1, 1000])
check("tăng dần", sol.shuffle(list(range(1, 9)), 4), [1, 5, 2, 6, 3, 7, 4, 8])

PASS  ví dụ 1: [2, 3, 5, 4, 1, 7]
PASS  ví dụ 2: [1, 4, 2, 3, 3, 2, 4, 1]
PASS  ví dụ 3: [1, 2, 1, 2]
PASS  n = 1: [7, 9]
PASS  giá trị biên 1000: [1000, 1, 1, 1000]
PASS  tăng dần: [1, 5, 2, 6, 3, 7, 4, 8]


### 13.1. Đối chiếu 5 cách — ngẫu nhiên có seed

Cách 5 sửa input nên ta truyền **bản sao** `nums[:]`.

In [18]:
import random
rng = random.Random(1470)
for _ in range(500):
    n = rng.randint(1, 500)
    nums = [rng.randint(1, 1000) for _ in range(2 * n)]
    exp = shuffle_two_pointers(nums, n)
    assert shuffle_index(nums, n) == exp
    assert shuffle_zip(nums, n) == exp
    assert shuffle_slices(nums, n) == exp
    assert shuffle_inplace(nums[:], n) == exp
    assert Solution().shuffle(nums, n) == exp
print("500 mảng ngẫu nhiên: 6 hàm cho cùng kết quả")

500 mảng ngẫu nhiên: 6 hàm cho cùng kết quả


## 14. Lỗi người mới hay gặp

### Lỗi 1 — Lấy `y` ở `nums[i + 1]` thay vì `nums[i + n]`

In [19]:
def wrong_neighbor(nums, n):
    ans = []
    for i in range(n):
        ans += [nums[i], nums[i + 1]]
    return ans

print("Sai:", wrong_neighbor([2, 5, 1, 3, 4, 7], 3), "| Đúng:", shuffle_index([2, 5, 1, 3, 4, 7], 3))

Sai: [2, 5, 5, 1, 1, 3] | Đúng: [2, 3, 5, 4, 1, 7]


**Vì sao sai:** `x_i` và `y_i` cách nhau `n` vị trí trong input, không phải 1. (`ans += [a, b]` nối list — tương đương hai lần `append`.)

### Lỗi 2 — Duyệt `range(2 * n)` → vượt chỉ số

In [20]:
try:
    nums, n = [2, 5, 1, 3, 4, 7], 3
    ans = []
    for i in range(2 * n):
        ans += [nums[i], nums[i + n]]
except IndexError as e:
    print("Lỗi:", e, "→ chỉ có n cặp, duyệt range(n)")

Lỗi: list index out of range → chỉ có n cặp, duyệt range(n)


### Lỗi 3 — Ghi ngay vào `nums` khi chưa đóng gói (in-place ngây thơ)

In [21]:
def wrong_inplace_naive(nums, n):
    nums = nums[:]
    src = nums                       # cùng một list! không phải bản sao
    for i in range(n):
        nums[2 * i], nums[2 * i + 1] = src[i], src[i + n]
    return nums

print("Sai:", wrong_inplace_naive([2, 5, 1, 3, 4, 7], 3), "| Đúng:", shuffle_index([2, 5, 1, 3, 4, 7], 3))

Sai: [2, 3, 3, 4, 3, 7] | Đúng: [2, 3, 5, 4, 1, 7]


**Vì sao sai:** `src = nums` chỉ tạo **tên mới** cho cùng list; ghi `nums[2i+1]` làm hỏng `x` chưa đọc (ví dụ `nums[1] = 5` bị ghi đè bằng `3` trước khi được dùng). **Sửa:** dùng mảng mới, hoặc kỹ thuật đóng gói ở cách 5.

### Lỗi 4 — Mở gói in-place theo chiều xuôi

In [22]:
def wrong_unpack_forward(nums, n):
    nums = nums[:]
    for i in range(n):
        nums[i] |= nums[i + n] << 10
    for i in range(n):                         # xuôi: sai
        y, x = nums[i] >> 10, nums[i] & 1023
        nums[2 * i + 1] = y
        nums[2 * i] = x
    return nums

print("Sai:", wrong_unpack_forward([2, 5, 1, 3, 4, 7], 3), "| Đúng:", shuffle_index([2, 5, 1, 3, 4, 7], 3))

Sai: [2, 3, 3, 0, 3, 0] | Đúng: [2, 3, 5, 4, 1, 7]


**Vì sao sai:** ở `i = 0` ta ghi ô `1` — nhưng ô `1` đang chứa gói `(x_1, y_1)` chưa mở. **Sửa:** mở từ `i = n-1` về `0`.

### Lỗi 5 — Dùng `|` khi các bit chồng nhau

`x | (y << 10)` chỉ đúng khi `x < 1024`. Nếu đề cho giá trị tới `10^6` thì phải dịch nhiều bit hơn (ví dụ `<< 20`). Luôn kiểm tra constraints trước khi dùng mẹo bit.

## 15. Mẹo nhận dạng pattern

- Input là **hai nửa ghép lại** và output **xen kẽ** → ánh xạ `i ↔ 2i`, `i + n ↔ 2i + 1`; hoặc `zip(nửa_đầu, nửa_sau)`.
- "Chẵn nhận A, lẻ nhận B" → gán slice `ans[::2]`, `ans[1::2]`.
- Follow-up "**không dùng bộ nhớ phụ**" và giá trị bị chặn nhỏ → **đóng gói hai giá trị vào một số** (bit hoặc `a + b * M`), rồi mở theo thứ tự không phá dữ liệu.
- Bài cùng họ: *Merge Strings Alternately* (1768), *Concatenation of Array* (1929), *Rearrange Array Elements by Sign* (2149), *Build Array from Permutation* (1920 — cũng có mẹo in-place mã hóa).

## 16. Biến thể

### 16.1. Phép ngược: un-shuffle `[x1,y1,...]` → `[x1..xn, y1..yn]`

In [23]:
def unshuffle(arr):
    return arr[::2] + arr[1::2]

print(unshuffle([2, 3, 5, 4, 1, 7]))   # [2, 5, 1, 3, 4, 7]

[2, 5, 1, 3, 4, 7]


### 16.2. Merge Strings Alternately (LeetCode 1768) — hai phần **khác độ dài**

In [24]:
def merge_alternately(w1, w2):
    out = []
    for i in range(max(len(w1), len(w2))):
        if i < len(w1):
            out.append(w1[i])
        if i < len(w2):
            out.append(w2[i])
    return "".join(out)

print(merge_alternately("abc", "pqr"), merge_alternately("ab", "pqrs"))   # apbqcr apbqrs

apbqcr apbqrs


### 16.3. Xen kẽ `k` phần bằng nhau

In [25]:
def shuffle_k(nums, k):
    m = len(nums) // k
    return [nums[part * m + i] for i in range(m) for part in range(k)]

print(shuffle_k([1, 2, 3, 4, 5, 6], 3))   # phần [1,2],[3,4],[5,6] → [1,3,5,2,4,6]

[1, 3, 5, 2, 4, 6]


## 17. Bài tập luyện tập (tăng dần)

**Bài 1 — Tính tay.** `nums = [9, 8, 7, 6, 5, 4, 3, 2]`, `n = 4`. Viết kết quả. Phần tử ở `ans[5]` lấy từ `nums[?]`?

**Bài 2 — Dự đoán output.**
```python
a = [1, 2, 3, 4, 5, 6]
print(a[::2], a[1::2], a[::-2], 6 << 10, (6 << 10) >> 10)
```

Bài 3–6 có cell TODO.

### Bài 3 — TODO: tự viết lời giải công thức chỉ số

In [26]:
def exercise_shuffle(nums, n):
    # TODO: ans = [0] * (2n); ans[2i] = nums[i]; ans[2i+1] = nums[i+n]
    pass

# assert exercise_shuffle([2,5,1,3,4,7], 3) == [2,3,5,4,1,7]

### Bài 4 — TODO: un-shuffle bằng vòng lặp (không dùng slicing có bước)

In [27]:
def exercise_unshuffle(arr):
    # TODO: n = len(arr) // 2; res[i] = arr[2i], res[i+n] = arr[2i+1]
    pass

# assert exercise_unshuffle([2,3,5,4,1,7]) == [2,5,1,3,4,7]

### Bài 5 — TODO: Rearrange Array Elements by Sign (LeetCode 2149)
`nums` có số dương và số âm **bằng nhau** về số lượng. Trả mảng xen kẽ `dương, âm, dương, âm, ...`, giữ nguyên thứ tự tương đối trong từng nhóm.

In [28]:
def exercise_rearrange_by_sign(nums):
    # TODO: hai con trỏ ghi: pos_idx = 0, neg_idx = 1, mỗi lần nhảy 2
    pass

# assert exercise_rearrange_by_sign([3,1,-2,-5,2,-4]) == [3,-2,1,-5,2,-4]
# assert exercise_rearrange_by_sign([-1,1]) == [1,-1]

### Bài 6 — Mini challenge: số lần shuffle để quay về ban đầu
Áp dụng `shuffle` liên tục lên `[1, 2, ..., 2n]`. Sau ít nhất bao nhiêu lần mảng quay về như cũ? (Gợi ý toán: vị trí `k` (0-indexed) đi tới `2k mod (2n-1)` với `k < 2n-1`, vị trí cuối đứng yên.)

In [29]:
def exercise_shuffle_period(n):
    # TODO cách 1: mô phỏng đến khi quay về
    # TODO cách 2: tìm t nhỏ nhất với 2^t ≡ 1 (mod 2n-1)   (n >= 2)
    pass

# assert exercise_shuffle_period(3) == 4
# assert exercise_shuffle_period(4) == 3

## 18. Gợi ý 3 mức

| Bài | Mức 1 — hướng nghĩ | Mức 2 — cấu trúc | Mức 3 — gần code |
|---|---|---|---|
| 3 | Bảng ánh xạ mục 4. | `for i in range(n)`, hai phép gán. | `ans[2*i], ans[2*i+1] = nums[i], nums[i+n]` |
| 4 | Đảo ngược ánh xạ: `2i → i`, `2i+1 → i+n`. | cấp phát `res = [0]*len(arr)`. | `res[i] = arr[2*i]; res[i+n] = arr[2*i+1]` |
| 5 | Dương vào ô chẵn, âm vào ô lẻ. | hai chỉ số ghi riêng. | `if x > 0: ans[p] = x; p += 2` / `else: ans[q] = x; q += 2` |
| 6 | Theo dõi một vị trí là đủ vì mọi vị trí cùng nhân 2. | vòng `while`. | `t, v = 1, 2 % (2n-1)`; `while v != 1: v = v*2 % (2n-1); t += 1` |

## 19. Đáp án tham khảo (chỉ xem sau khi tự làm)

**Bài 1:** `x = [9,8,7,6]`, `y = [5,4,3,2]` → `[9,5,8,4,7,3,6,2]`; `ans[5]` (lẻ) lấy từ `nums[5 // 2 + 4] = nums[6] = 3`.
**Bài 2:** `[1, 3, 5] [2, 4, 6] [6, 4, 2] 6144 6`.

In [30]:
def answer_shuffle(nums, n):
    ans = [0] * (2 * n)
    for i in range(n):
        ans[2 * i], ans[2 * i + 1] = nums[i], nums[i + n]
    return ans

def answer_unshuffle(arr):
    n = len(arr) // 2
    res = [0] * len(arr)
    for i in range(n):
        res[i] = arr[2 * i]
        res[i + n] = arr[2 * i + 1]
    return res

def answer_rearrange_by_sign(nums):
    ans = [0] * len(nums)
    p, q = 0, 1
    for x in nums:
        if x > 0:
            ans[p] = x; p += 2
        else:
            ans[q] = x; q += 2
    return ans

def answer_shuffle_period(n):
    if n == 1:
        return 1
    m = 2 * n - 1
    t, v = 1, 2 % m
    while v != 1:
        v = v * 2 % m
        t += 1
    return t

def simulate_period(n):
    start = list(range(1, 2 * n + 1))
    cur, t = answer_shuffle(start, n), 1
    while cur != start:
        cur, t = answer_shuffle(cur, n), t + 1
    return t

assert answer_shuffle([2,5,1,3,4,7], 3) == [2,3,5,4,1,7]
assert answer_unshuffle([2,3,5,4,1,7]) == [2,5,1,3,4,7]
assert answer_rearrange_by_sign([3,1,-2,-5,2,-4]) == [3,-2,1,-5,2,-4]
assert answer_rearrange_by_sign([-1,1]) == [1,-1]
assert answer_shuffle_period(3) == 4
assert answer_shuffle_period(4) == 3
for n in range(1, 60):
    assert answer_shuffle_period(n) == simulate_period(n), n
print("Bài 1:", answer_shuffle([9, 8, 7, 6, 5, 4, 3, 2], 4))
a = [1, 2, 3, 4, 5, 6]
print("Bài 2:", a[::2], a[1::2], a[::-2], 6 << 10, (6 << 10) >> 10)
print("Đáp án bài 3–6: PASS (bài 6 khớp mô phỏng với n = 1..59)")

Bài 1: [9, 5, 8, 4, 7, 3, 6, 2]
Bài 2: [1, 3, 5] [2, 4, 6] [6, 4, 2] 6144 6
Đáp án bài 3–6: PASS (bài 6 khớp mô phỏng với n = 1..59)


## 20. Template nhớ nhanh + checklist submit

```python
# Hai nửa [A | B] → xen kẽ A0 B0 A1 B1 ...
ans = [0] * (2 * n)
for i in range(n):
    ans[2 * i] = nums[i]          # nửa đầu → ô chẵn
    ans[2 * i + 1] = nums[i + n]  # nửa sau → ô lẻ
# hoặc: ans[::2], ans[1::2] = nums[:n], nums[n:]
```

**Checklist trước khi nộp**
- [ ] `y_i` ở `nums[i + n]`, không phải `nums[i + 1]`.
- [ ] Duyệt `range(n)` (n cặp), không phải `range(2n)`.
- [ ] Mảng kết quả đủ `2n` ô.
- [ ] Nếu in-place: đóng gói trước, mở **từ cuối về đầu**, kiểm tra giới hạn giá trị.
- [ ] Method tên `shuffle(nums, n)`, trả về list.